# 03. Calcul des isochrones

Calcul des isochrones de temps de parcours piéton et cyclable en routage
réel (OSMnx) depuis chaque arrêt de transport en commun structurant, sur
le périmètre de la Métropole Rouen Normandie.

**Prérequis** : exécuter `01_acquisition_donnees_OSM.ipynb` et `02_arrets_TC.ipynb` au préalable  
**Entrées** :
- `data/reseau_pieton_MRN.graphml` — graphe piéton (topologie NetworkX, EPSG:2154)
- `data/reseau_velo_MRN.graphml` — graphe cyclable (topologie NetworkX, EPSG:2154)
- `data/arrets_2026.gpkg` — arrêts TC structurants (situation 2026)

**Constantes** (référence : document de cadrage) :
- Vitesse à pied : 5 km/h
- Vitesse à vélo : 15 km/h
- Seuil d'accessibilité : 15 minutes
- Pas des tranches : 5 minutes

**Projection** : EPSG:2154 (RGF93 / Lambert-93)  
**Sortie** : *(définie dans une étape ultérieure)*

## 1. Imports et paramètres

In [ ]:
from pathlib import Path

import geopandas as gpd
import networkx as nx
import osmnx as ox

print(f"OSMnx version : {ox.__version__}")

# Vitesses de déplacement (km/h) — référence Cerema / INSEE et cycliste urbain ordinaire
VITESSE_PIED = 5
VITESSE_VELO = 15

# Paramètres d'accessibilité (minutes)
SEUIL_ACCESSIBILITE = 15
PAS_TRANCHE = 5

CRS_PROJET = "EPSG:2154"

Détection de la racine du projet via le fichier sentinelle `.projectroot`,
puis ancrage de tous les chemins dessus. Cette approche est indépendante du
répertoire de travail : elle fonctionne sous JupyterLab comme sous VSCode,
quel que soit le réglage `notebookFileRoot`, et après un clone comme après un
téléchargement ZIP du dépôt.

In [ ]:
def trouver_racine(marqueur=".projectroot"):
    """Remonte depuis le cwd jusqu'au dossier contenant `marqueur`."""
    base = Path.cwd().resolve()
    for parent in (base, *base.parents):
        if (parent / marqueur).exists():
            return parent
    raise FileNotFoundError(f"Racine introuvable (marqueur {marqueur!r})")


ROOT = trouver_racine()
DATA_DIR = ROOT / "data"
print(f"Racine du projet : {ROOT}")

## 2. Chargement des graphes de réseau

Les graphes piéton et cyclable produits par `01_acquisition_donnees_OSM.ipynb`
sont rechargés depuis leurs fichiers `.graphml`, qui conservent la topologie
nécessaire au calcul des plus courts chemins (étapes suivantes).

In [ ]:
G_pieton = ox.load_graphml(DATA_DIR / "reseau_pieton_MRN.graphml")
G_velo = ox.load_graphml(DATA_DIR / "reseau_velo_MRN.graphml")

Contrôle de cohérence : nombre de nœuds/arêtes et système de projection de
chaque graphe. Les deux graphes doivent être projetés en EPSG:2154.

In [ ]:
for nom, G in {"piéton": G_pieton, "vélo": G_velo}.items():
    print(
        f"Réseau {nom:<7} | "
        f"{G.number_of_nodes():>7,} nœuds | "
        f"{G.number_of_edges():>7,} arêtes | "
        f"CRS : {G.graph.get('crs')}"
    )

## 3. Chargement des arrêts de transport en commun

Couche d'arrêts TC structurants pour la situation 2026, produite par
`02_arrets_TC.ipynb`.

In [ ]:
arrets = gpd.read_file(DATA_DIR / "arrets_2026.gpkg")

print(f"{len(arrets):,} arrêts chargés")
print(f"CRS : {arrets.crs}")
arrets.head()

Contrôle de cohérence : la couche des arrêts doit partager le système de
projection des graphes (EPSG:2154) avant tout rattachement spatial des
arrêts au réseau (étape suivante).

In [ ]:
assert arrets.crs is not None, "La couche des arrêts n'a pas de CRS défini."
assert arrets.crs.to_epsg() == 2154, (
    f"CRS inattendu : {arrets.crs.to_epsg()} (attendu : 2154)"
)
print("Arrêts et graphes alignés en EPSG:2154.")